# cAPTure: graph OOF score-saturation audit

This notebook investigates why a model can have near-perfect OOF ranking while producing no alerts at every frozen operational budget. It is bound to the immutable four-model comparison and reads the same checksum-verified development OOF predictions.

The audit does not train, rescore, recalibrate, or alter thresholds. It does not access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

## 1. Prepare the Colab environment

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import Markdown, display
from utils.capture_graph_score_saturation import (
    run_graph_score_saturation_audit,
    validate_graph_score_saturation_audit,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("A GPU runtime is not required for this audit.")

Mounted at /content/drive
Repository commit: 7be2bf6310e7c98f4441d4c4d16c88b4b10f6679
A GPU runtime is not required for this audit.


## 2. Bind the immutable comparison and OOF artifacts

In [2]:
BINDING_RUN_ID = "20260929T144234_934044Z_graph_training_binding"
COMPARISON_RUN_ID = "20261001T153351_608797Z_graph_four_model_comparison"
TRAINING_ROOT = DRIVE_ROOT / "graph_training_runs" / BINDING_RUN_ID
COMPARISON_DIR = DRIVE_ROOT / "graph_comparison_runs" / COMPARISON_RUN_ID

for path in (TRAINING_ROOT, COMPARISON_DIR):
    if not path.is_dir():
        raise FileNotFoundError(f"Required audit input is missing: {path}")
comparison_report = json.loads(
    (COMPARISON_DIR / "comparison_report.json").read_text(encoding="utf-8")
)
threshold_rows = []
for model_name, model_report in comparison_report["models"].items():
    for budget_name, item in model_report["thresholds"].items():
        threshold_rows.append({
            "model": model_name,
            "budget": budget_name,
            "threshold": item["threshold"],
            "threshold_above_one": item["threshold"] > 1.0,
            "worst_fold_false_alert_windows_per_hour": item["worst_fold_false_alert_windows_per_hour"],
        })
display(pd.DataFrame(threshold_rows).set_index(["model", "budget"]))
print("Source comparison:", COMPARISON_DIR)

threshold  threshold_above_one  \
model      budget                                              
edge_mlp   one_per_hour        0.999966                False   
           one_per_12_hours    0.999966                False   
           one_per_5_minutes   0.937334                False   
edge_gru   one_per_hour        1.000000                 True   
           one_per_12_hours    1.000000                 True   
           one_per_5_minutes   1.000000                 True   
static_gnn one_per_hour        0.999141                False   
           one_per_12_hours    1.000000                False   
           one_per_5_minutes   0.966401                False   
st_gnn     one_per_hour        1.000000                 True   
           one_per_12_hours    1.000000                 True   
           one_per_5_minutes   1.000000                 True   

                              worst_fold_false_alert_windows_per_hour  
model      budget                                                      
edge_mlp   one_per_hour                                      0.000000  
           one_per_12_hours                                  0.000000  
           one_per_5_minutes                                 8.593168  
edge_gru   one_per_hour                                      0.000000  
           one_per_12_hours                                  0.000000  
           one_per_5_minutes                                 0.000000  
static_gnn one_per_hour                                      0.946445  
           one_per_12_hours                                  0.000000  
           one_per_5_minutes                                11.970775  
st_gnn     one_per_hour                                      0.000000  
           one_per_12_hours                                  0.000000  
           one_per_5_minutes                                 0.000000

Source comparison: /content/drive/MyDrive/capture_gate0/graph_comparison_runs/20261001T153351_608797Z_graph_four_model_comparison


## 3. Run or verify the read-only audit

The audit verifies the source comparison, completion files, metrics, and OOF checksums before scanning score tails. Set `RUN_AUDIT=True` after reviewing the bound thresholds. A completed audit is immutable and can be reloaded by setting its run ID.

In [4]:
AUDIT_RUN_ID = None  # Set this to reuse a completed audit.
RUN_AUDIT = True
if AUDIT_RUN_ID is None:
    AUDIT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_score_saturation_audit"
    )
OUTPUT_DIR = (
    DRIVE_ROOT / "graph_score_saturation_audit_runs" / AUDIT_RUN_ID
)
print("Saturation-audit run ID:", AUDIT_RUN_ID)
print("Audit output:", OUTPUT_DIR)

if OUTPUT_DIR.exists():
    audit_report = validate_graph_score_saturation_audit(
        comparison_dir=COMPARISON_DIR,
        output_dir=OUTPUT_DIR,
    )
elif RUN_AUDIT:
    audit_report = run_graph_score_saturation_audit(
        comparison_dir=COMPARISON_DIR,
        training_root=TRAINING_ROOT,
        output_dir=OUTPUT_DIR,
    )
else:
    audit_report = None
    print("Set RUN_AUDIT=True to start the read-only saturation audit.")

Saturation-audit run ID: 20261001T175306_830261Z_graph_score_saturation_audit
Audit output: /content/drive/MyDrive/capture_gate0/graph_score_saturation_audit_runs/20261001T175306_830261Z_graph_score_saturation_audit
Auditing score tails for edge_mlp...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Auditing score tails for edge_gru...
Auditing score tails for static_gnn...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Auditing score tails for st_gnn...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. Review the model-level diagnosis

A confirmed blocker requires all three signals: a selected threshold above one, benign windows stored exactly at one whose smallest attainable positive rate exceeds the budget, and attack packets tied at the same stored score.

In [5]:
if audit_report is None:
    raise RuntimeError("Run or load the audit before displaying results.")
model_summary = pd.DataFrame(audit_report["tables"]["model_summary"])
display(model_summary.set_index("model_display")[[
    "stored_score_type", "primary_threshold",
    "primary_threshold_above_one",
    "primary_worst_fold_false_alert_windows_per_hour",
    "minimum_positive_worst_fold_false_alert_windows_per_hour",
    "exact_one_benign_windows", "exact_one_attack_packets",
    "exact_one_attack_packet_fraction", "primary_diagnosis",
    "oof_rescoring_from_checkpoint_recommended",
]])

,stored_score_type,primary_threshold,primary_threshold_above_one,primary_worst_fold_false_alert_windows_per_hour,minimum_positive_worst_fold_false_alert_windows_per_hour,exact_one_benign_windows,exact_one_attack_packets,exact_one_attack_packet_fraction,primary_diagnosis,oof_rescoring_from_checkpoint_recommended
model_display,,,,,,,,,,
Edge MLP,float,0.999966,False,0.000000,2.056377,0,0,0.000000,no_exact_one_operational_blocker_detected,False
EdgeGRU,float,1.000000,True,0.000000,49.959866,1324,5607581,0.737767,saturated_top_tie_forces_zero_alert_threshold,True
StaticGNN,float,0.999141,False,0.946445,0.113436,0,24799,0.003263,no_exact_one_operational_blocker_detected,False
ST-GNN,float,1.000000,True,0.000000,25.445042,667,13748,0.001809,saturated_top_tie_forces_zero_alert_threshold,True


## 5. Review every frozen false-alert budget

In [6]:
budget_diagnostics = pd.DataFrame(
    audit_report["tables"]["budget_diagnostics"]
)
display(budget_diagnostics.set_index(["model_display", "budget"])[[
    "target_false_alert_windows_per_hour", "selected_threshold",
    "selected_threshold_above_one",
    "reported_worst_fold_false_alert_windows_per_hour",
    "minimum_positive_worst_fold_false_alert_windows_per_hour",
    "top_score_block_within_budget",
    "shared_exact_one_benign_attack_tie", "diagnosis",
]])

target_false_alert_windows_per_hour  \
model_display budget                                                   
Edge MLP      one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   
EdgeGRU       one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   
StaticGNN     one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   
ST-GNN        one_per_hour                                  1.000000   
              one_per_12_hours                              0.083333   
              one_per_5_minutes                            12.000000   

                                 selected_threshold  \
model_display budget                                  
Edge MLP      one_per_hour                 0.999966   
              one_per_12_hours             0.999966   
              one_per_5_minutes            0.937334   
EdgeGRU       one_per_hour                 1.000000   
              one_per_12_hours             1.000000   
              one_per_5_minutes            1.000000   
StaticGNN     one_per_hour                 0.999141   
              one_per_12_hours             1.000000   
              one_per_5_minutes            0.966401   
ST-GNN        one_per_hour                 1.000000   
              one_per_12_hours             1.000000   
              one_per_5_minutes            1.000000   

                                 selected_threshold_above_one  \
model_display budget                                            
Edge MLP      one_per_hour                              False   
              one_per_12_hours                          False   
              one_per_5_minutes                         False   
EdgeGRU       one_per_hour                               True   
              one_per_12_hours                           True   
              one_per_5_minutes                          True   
StaticGNN     one_per_hour                              False   
              one_per_12_hours                          False   
              one_per_5_minutes                         False   
ST-GNN        one_per_hour                               True   
              one_per_12_hours                           True   
              one_per_5_minutes                          True   

                                 reported_worst_fold_false_alert_windows_per_hour  \
model_display budget                                                                
Edge MLP      one_per_hour                                               0.000000   
              one_per_12_hours                                           0.000000   
              one_per_5_minutes                                          8.593168   
EdgeGRU       one_per_hour                                               0.000000   
              one_per_12_hours                                           0.000000   
              one_per_5_minutes                                          0.000000   
StaticGNN     one_per_hour                                               0.946445   
              one_per_12_hours                                           0.000000   
              one_per_5_minutes                                         11.970775   
ST-GNN        one_per_hour                                               0.000000   
              one_per_12_hours                                           0.000000   
              one_per_5_minutes                                          0.000000   

                                 minimum_positive_worst_fold_false_alert_windows_per_hour  \
model_display budget                                      

## 6. Inspect packet and window score tails

The detailed tables retain scenario and fold boundaries. Focus on the exact-one counts, maximum-tie sizes, distinct-score fractions, and upper quantiles.

In [7]:
packet_tails = pd.DataFrame(audit_report["tables"]["packet_score_tails"])
window_tails = pd.DataFrame(
    audit_report["tables"]["window_maximum_score_tails"]
)
packet_columns = [
    "packets", "distinct_scores", "distinct_score_fraction",
    "score_q99", "score_q999", "score_q9999", "score_q99999",
    "maximum_score", "exact_one_scores", "exact_one_score_fraction",
    "maximum_tie_scores",
]
window_columns = [
    "wall_clock_windows", "observed_windows", "empty_windows",
    "window_maximum_score_q99", "window_maximum_score_q999",
    "window_maximum_score_q9999", "window_maximum_score_q99999",
    "maximum_window_maximum_score", "exact_one_window_maxima",
    "maximum_tie_windows",
]
display(packet_tails.set_index(
    ["model_display", "fold", "scenario", "packet_class"]
)[packet_columns])
display(window_tails.set_index(
    ["model_display", "fold", "scenario", "window_class"]
)[window_columns])

packets  distinct_scores  \
model_display fold scenario          packet_class                             
Edge MLP      A    train_dollar_char benign        1705003            10267   
                                     attack        1177552             1026   
                   train_slash_char  benign        1705003            10412   
                                     attack        4720513             2186   
                   train_sub_exf     benign        1705003            10031   
                                     attack         520804             1389   
              B    train_empty_conn  benign         746806             1798   
                                     attack         428973              854   
                   train_qos_mid     benign         746806             1837   
                                     attack         752911              661   
EdgeGRU       A    train_dollar_char benign        1705003           367064   
                                     attack        1177552            15354   
                   train_slash_char  benign        1705003           367633   
                                     attack        4720513              394   
                   train_sub_exf     benign        1705003           367578   
                                     attack         520804              648   
              B    train_empty_conn  benign         746806           335585   
                                     attack         428973            38673   
                   train_qos_mid     benign         746806           335470   
                                     attack         752911           106735   
StaticGNN     A    train_dollar_char benign        1705003           406166   
                                     attack        1177552            25712   
                   train_slash_char  benign        1705003           406048   
                                     attack        4720513            27702   
                   train_sub_exf     benign        1705003           405678   
                                     attack         520804            14478   
              B    train_empty_conn  benign         746806           355355   
                                     attack         428973            35516   
                   train_qos_mid     benign         746806           355718   
                                     attack         752911            75636   
ST-GNN        A    train_dollar_char benign        1705003           326857   
                                     attack        1177552             8402   
                   train_slash_char  benign        1705003           327282   
                                     attack        4720513           153868   
                   train_sub_exf     benign        1705003           327763   
                                     attack         520804            11748   
              B    train_empty_conn  benign         746806           334425   
                                     attack         428973            37399   
                   train_qos_mid     benign         746806           335431   
                                     attack         752911            43212   

                                                   distinct_score_fraction  \
model_display fold scenario          packet_class                            
Edge MLP      A    train_dollar_char benign                       0.006022   
                                     attack                       0.000871   
                   train_slash_char  benign                       0.006107   
                                     attack                       0.000463   
                   train_sub_exf     benign                       0.005883   
                                     attack                       0.002667   
              B    train_empty_conn  benign                       0.002408   
                              

wall_clock_windows  \
model_display fold scenario          window_class                            
Edge MLP      A    train_dollar_char benign                           5813   
                                     attack_containing                3547   
                   train_slash_char  benign                           6630   
                                     attack_containing                2730   
                   train_sub_exf     benign                           6676   
                                     attack_containing                2684   
              B    train_empty_conn  benign                           4068   
                                     attack_containing                5292   
                   train_qos_mid     benign                           4404   
                                     attack_containing                4956   
EdgeGRU       A    train_dollar_char benign                           5813   
                                     attack_containing                3547   
                   train_slash_char  benign                           6630   
                                     attack_containing                2730   
                   train_sub_exf     benign                           6676   
                                     attack_containing                2684   
              B    train_empty_conn  benign                           4068   
                                     attack_containing                5292   
                   train_qos_mid     benign                           4404   
                                     attack_containing                4956   
StaticGNN     A    train_dollar_char benign                           5813   
                                     attack_containing                3547   
                   train_slash_char  benign                           6630   
                                     attack_containing                2730   
                   train_sub_exf     benign                           6676   
                                     attack_containing                2684   
              B    train_empty_conn  benign                           4068   
                                     attack_containing                5292   
                   train_qos_mid     benign                           4404   
                                     attack_containing                4956   
ST-GNN        A    train_dollar_char benign                           5813   
                                     attack_containing                3547   
                   train_slash_char  benign                           6630   
                                     attack_containing                2730   
                   train_sub_exf     benign                           6676   
                                     attack_containing                2684   
              B    train_empty_conn  benign                           4068   
                                     attack_containing                5292   
                   train_qos_mid     benign                           4404   
                                     attack_containing                4956   

                                                        observed_windows  \
model_display fold scenario          window_class                          
Edge MLP      A    train_dollar_char benign                         5569   
                                     attack_containing              3547   
                   train_slash_char  benign                         6386   
                                     attack_containing              2730   
                   train_sub_exf     benign                         6431   
                                     attack_containing              2684   
              B    train_empty_conn  benign                         4068   
                                     attack_containing              5292   
                   

## 7. Review the saved audit report

The audit preserves the original comparison as evidence. If rescoring is recommended, the next artifact must repeat only OOF inference from the saved final checkpoints and store raw logits or float64 sigmoid scores under a new versioned contract.

In [8]:
display(Markdown((OUTPUT_DIR / "review.md").read_text(encoding="utf-8")))
print("Audit status:", audit_report["status"])
print("Saturated models:", audit_report["saturated_models"])
print("OOF rescoring recommended:", audit_report["oof_rescoring_recommended"])
print("Threshold policy changed:", audit_report["threshold_policy_changed"])
print("Model training performed:", audit_report["model_training_performed"])
print("Held-out scenarios accessed:", audit_report["held_out_scenarios_accessed"])
print("Saved audit:", OUTPUT_DIR / "score_saturation_audit.json")
print("Saved review:", OUTPUT_DIR / "review.md")

# cAPTure graph score-saturation audit

Status: **score_saturation_confirmed**.

Score saturation is confirmed for EdgeGRU, ST-GNN. Their thresholded operational metrics must not be interpreted as model failures. Re-run OOF inference from the immutable final checkpoints and preserve raw logits or float64 sigmoid scores.

## Primary-budget diagnosis

| Model | Stored score | Threshold | Threshold > 1 | Benign windows at 1 | Attack packets at 1 | Minimum positive worst-fold FA/h | Diagnosis |
|---|---|---|---|---|---|---|---|
| Edge MLP | float | 0.999966 | False | 0 | 0 | 2.05638 | no_exact_one_operational_blocker_detected |
| EdgeGRU | float | 1 | True | 1324 | 0.737767 | 49.9599 | saturated_top_tie_forces_zero_alert_threshold |
| StaticGNN | float | 0.999141 | False | 0 | 0.0032627 | 0.113436 | no_exact_one_operational_blocker_detected |
| ST-GNN | float | 1 | True | 667 | 0.00180877 | 25.445 | saturated_top_tie_forces_zero_alert_threshold |

## Interpretation

A threshold above one is a valid consequence of the frozen `score >= threshold` and `nextafter` tie policy when the benign top-score block is too large for the budget. It is not evidence that the threshold search malfunctioned. When attack scores share the same stored float32 value of one, probability artifacts no longer contain enough resolution to separate that tie.

Ranking metrics remain valid for the stored score ordering, but operational metrics from a degenerate threshold are not suitable for comparing detection utility. Existing OOF files and the four-model comparison remain immutable evidence; any corrected evaluation must use new versioned artifacts.

## Next action

If rescoring is recommended, load each saved final checkpoint and repeat only outer-fold inference. Store raw logits and a float64 sigmoid derived before serialization, verify identical packet keys, and run a versioned operational comparison. Do not retrain, refit, or access held-out scenarios.

Test1, Test2, `train_pub_exf`, and `train_user_prop` were not accessed.


Audit status: score_saturation_confirmed
Saturated models: ['edge_gru', 'st_gnn']
OOF rescoring recommended: True
Threshold policy changed: False
Model training performed: False
Held-out scenarios accessed: False
Saved audit: /content/drive/MyDrive/capture_gate0/graph_score_saturation_audit_runs/20261001T175306_830261Z_graph_score_saturation_audit/score_saturation_audit.json
Saved review: /content/drive/MyDrive/capture_gate0/graph_score_saturation_audit_runs/20261001T175306_830261Z_graph_score_saturation_audit/review.md
